# CellPatcher

CellPatcher helps you extract per-cell image patches from paired image and mask datasets, even when data is multi-dimensional (for example channels, z-stacks, or timepoints).
The workflow keeps full non-spatial dimensions in the exported patches while providing visual previews through projected views for easier validation.

## Workflow Overview

1. Validate image and mask folders and confirm file matching.
2. Configure axis mapping for image and mask arrays.
3. Select the mask channel used for cell labeling (when applicable).
4. Preview detected cells and scan dataset-wide bounding-box sizes.
5. Configure patch extraction and resize parameters.
6. Run batch patch extraction with optional parallel workers.

## Notes

- Border-touching cells are excluded to avoid partial crops.
- Parallel execution is available for scanning and patch export when dependencies are installed.
- Suggested patch size is inferred automatically from the dataset scan and can be overridden.

In [ ]:
# @markdown Run this cell to check this notebook's version.

current_version = "0.0.1"
notebook_name = "CellPatcher"  # Replace with the actual notebook name

# First of all check if ipywidgets is installed, if not through an informative error
try:
    import ipywidgets as widgets
except ImportError:
    raise ImportError("ipywidgets is not installed. Please install it using 'pip install ipywidgets' or 'conda install -c conda-forge ipywidgets'.")

from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    
    try:
        import yaml
    except ImportError:
        raise ImportError("pyyaml is not installed. Please install it using 'pip install pyyaml' or 'conda install -c conda-forge pyyaml'.")

    import requests
    import base64

    # Create widgets
    github_repo = widgets.Text(
        value="",
        description="*GitHub Repository URL:",
        placeholder="e.g https://github.com/username/repository",
        layout=widgets.Layout(width="70%"),
        style={'description_width': '150px'},
    )

    github_token = widgets.Password(
        value="",
        description="*Personal Access Token:",
        placeholder="[Disabled] e.g. ghp_XXXXXXXXXXXXXXXXXXXX",
        disabled=True,
        layout=widgets.Layout(width="70%"),
        style={'description_width': '150px'},
    )

    test_button = widgets.Button(
        description="Test Connection",
        button_style='success',
    )

    output_area = widgets.HTML()

    # By default the repository is considered public
    if "repository_is_private" not in globals():
        repository_is_private = False
    elif repository_is_private:
        github_token.disabled = False
        github_token.placeholder = "e.g. ghp_XXXXXXXXXXXXXXXXXXXX"

    def on_test_button_clicked(b):
        global repository_is_private
        
        # Check that the GitHub repository URL is provided
        if not github_repo.value:
            output_area.value = "⚠️ Please provide a GitHub repository URL."
            return
        
        # Get the owner and repo name from the URL
        repo_url = github_repo.value.rstrip('/')
        github_owner, github_repo_name = repo_url.split('/')[-2:]
        
        # Online version checking file path
        version_file_path = "notebooks/notebook_latest_versions.yaml"
        version_url = f"https://api.github.com/repos/{github_owner}/{github_repo_name}/contents/{version_file_path}"

        # Do an initial request to check if the repository is public
        if not repository_is_private:
            version_response = requests.get(version_url)
            if version_response.status_code == 404:
                repository_is_private = True
                github_token.disabled = False
                github_token.placeholder = "e.g. ghp_XXXXXXXXXXXXXXXXXXXX"
                output_area.value = f"⚠️ We have detected that the repository might be private. Please provide a Personal Access Token and click 'Test Connection' again."
                return
        else:
            headers = {"Accept": "application/vnd.github.v3+json"}
            if not github_token.value:
                output_area.value = "⚠️ Personal Access Token is required for private repositories."
                return
            headers["Authorization"] = f"token {github_token.value}"

            version_response = requests.get(version_url, headers=headers)

        # Check the response status
        if version_response.status_code == 200:
            content = version_response.json()['content']
            decoded_content = base64.b64decode(content).decode('utf-8')
            config = yaml.safe_load(decoded_content)
            latest_version = config.get(notebook_name, "")

            output_area.value = (f"<b>Notebook version:</b> `{current_version}`<br>"
                                f"<b>Latest version available:</b> `{latest_version}`<br>")

            if latest_version == "":
                output_area.value += "⚠️ This notebook is not listed in the version file.<br>"
            elif current_version == latest_version:
                output_area.value += "✅ This notebook is up-to-date.<br>"
            else:
                output_area.value += f"⚠️ A new version of this notebook is available."
        else:
            output_area.value += "⚠️ Could not retrieve the version file.<br>"

    test_button.on_click(on_test_button_clicked)
    # Widget layout
    widget_box = widgets.VBox([github_repo, github_token, test_button, output_area])
    display(widget_box)
except ImportError:
    display(widgets.HTML('To check the notebook version, please go to the <a href="../Welcome.ipynb" target="_blank" style="color: #0066cc; text-decoration: underline;">Welcome notebook</a>.'))

In [ ]:
# @markdown Run this cell to import utility functions for the rest of the notebook. If this cell fails, the rest of the notebook will not work correctly.

from IPython.display import display
from ipywidgets import widgets
from pathlib import Path
import io
import os
import numpy as np

# Tkinter is optional in some notebook setups; keep a graceful fallback.
try:
    import tkinter as tk
    from tkinter import filedialog
    _HAS_TK = True
except Exception:
    _HAS_TK = False

# Optional dependencies are imported once here and reused by all cells.
try:
    import matplotlib.pyplot as plt
    _HAS_MPL = True
except Exception:
    plt = None
    _HAS_MPL = False

try:
    from PIL import Image
    _HAS_PIL = True
except Exception:
    Image = None
    _HAS_PIL = False

try:
    from scipy.ndimage import zoom as scipy_zoom
    _HAS_SCIPY = True
except Exception:
    scipy_zoom = None
    _HAS_SCIPY = False

try:
    from tqdm.auto import tqdm
    _HAS_TQDM = True
except Exception:
    tqdm = None
    _HAS_TQDM = False

try:
    import joblib
    _HAS_JOBLIB = True
except Exception:
    joblib = None
    _HAS_JOBLIB = False

try:
    import tifffile
    _HAS_TIFFFILE = True
except Exception:
    tifffile = None
    _HAS_TIFFFILE = False

try:
    import imageio.v3 as iio
    _HAS_IMAGEIO = True
except Exception:
    iio = None
    _HAS_IMAGEIO = False

# Define supported image file extensions for validation.
SUPPORTED_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".gif"}


def _browse_path(select="directory", title="Choose a path"):
    if not _HAS_TK:
        return ""

    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    try:
        # Use the appropriate native picker for files vs directories.
        if select == "file":
            selected_path = filedialog.askopenfilename(title=title)
        else:
            selected_path = filedialog.askdirectory(title=title, mustexist=False)
    finally:
        root.destroy()

    return selected_path or ""


def get_path_widget(
    description="Path:",
    placeholder="Type a path or click Browse",
    select="directory",
    default_value="",
    widgets_style=None,
    widgets_layout=None,
    browse_button_text="Browse",
):
    widgets_style = widgets_style or {}
    widgets_layout = widgets_layout or widgets.Layout(width="700px")

    path_text = widgets.Text(
        value=default_value,
        placeholder=placeholder,
        description=description,
        disabled=False,
        style=widgets_style,
        layout=widgets.Layout(width="80%"),
    )

    browse_button = widgets.Button(
        description=browse_button_text,
        icon="folder-open",
        tooltip="Open file picker",
        layout=widgets.Layout(width="20%"),
    )
    # Disable browse when Tk is unavailable; typing remains available.
    browse_button.disabled = not _HAS_TK

    status = widgets.HTML()

    def update_status(_=None):
        value = path_text.value.strip()
        if not value:
            status.value = '<span style="color:#777;">No path selected.</span>'
            return

        # Expand ~ and validate path type/existence for immediate feedback.
        resolved = Path(value).expanduser()
        if resolved.exists():
            if select == "file" and not resolved.is_file():
                status.value = f'<span style="color:#b26a00;">Path exists but is not a file: {resolved}</span>'
            elif select == "directory" and not resolved.is_dir():
                status.value = f'<span style="color:#b26a00;">Path exists but is not a directory: {resolved}</span>'
            else:
                status.value = f'<span style="color:#0a7f2e;"><b>Valid path</b>: {resolved}</span>'
        else:
            status.value = f'<span style="color:#b26a00;">Path does not exist yet: {resolved}</span>'

    def on_browse_clicked(_):
        selected = _browse_path(select=select, title=description)
        if selected:
            path_text.value = selected

    # Keep status synced with both manual input and browse selections.
    path_text.observe(update_status, names="value")
    browse_button.on_click(on_browse_clicked)

    row = widgets.HBox([path_text, browse_button], layout=widgets_layout)
    container = widgets.VBox([row, status])

    update_status()
    return container, path_text


def _list_supported_files(folder_path):
    folder = Path(folder_path).expanduser()
    return sorted(
        path for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
    )


def _find_candidate_matches(image_files, mask_files):
    mask_names = {mask_file: mask_file.stem.lower() for mask_file in mask_files}
    candidate_matches = {}

    # Start with permissive substring matching so suffix conventions can vary.
    for image_file in image_files:
        image_key = image_file.stem.lower()
        candidate_matches[image_file] = [
            mask_file
            for mask_file, mask_name in mask_names.items()
            if image_key in mask_name
        ]

    return candidate_matches


def _filter_shared_matches(candidate_matches):
    filtered_matches = {
        image_file: list(mask_matches)
        for image_file, mask_matches in candidate_matches.items()
    }

    # If an image has several candidate masks, drop those that are also claimed by other images.
    changed = True
    while changed:
        changed = False
        mask_to_images = {}
        for image_file, mask_matches in filtered_matches.items():
            for mask_file in mask_matches:
                mask_to_images.setdefault(mask_file, []).append(image_file)

        for image_file, mask_matches in filtered_matches.items():
            if len(mask_matches) <= 1:
                continue

            exclusive_matches = [
                mask_file
                for mask_file in mask_matches
                if len(mask_to_images.get(mask_file, [])) == 1
            ]

            if exclusive_matches and len(exclusive_matches) != len(mask_matches):
                filtered_matches[image_file] = exclusive_matches
                changed = True

    return filtered_matches


def validate_image_and_mask_folders(image_folder_path, mask_folder_path):
    if not image_folder_path.strip():
        raise ValueError("Please provide a folder for the images.")
    if not mask_folder_path.strip():
        raise ValueError("Please provide a folder for the masks.")

    image_folder = Path(image_folder_path).expanduser()
    mask_folder = Path(mask_folder_path).expanduser()

    if not image_folder.exists() or not image_folder.is_dir():
        raise ValueError(f"Images folder does not exist or is not a directory: {image_folder}")
    if not mask_folder.exists() or not mask_folder.is_dir():
        raise ValueError(f"Masks folder does not exist or is not a directory: {mask_folder}")

    image_files = _list_supported_files(image_folder)
    mask_files = _list_supported_files(mask_folder)

    if not image_files:
        raise ValueError(
            f"No supported image files were found in the images folder: {image_folder} "
            f"Supported extensions: {sorted(SUPPORTED_EXTENSIONS)}"
        )
    if not mask_files:
        raise ValueError(
            f"No supported image files were found in the masks folder: {mask_folder} "
            f"Supported extensions: {sorted(SUPPORTED_EXTENSIONS)}"
        )

    candidate_matches = _find_candidate_matches(image_files, mask_files)
    matched_pairs = _filter_shared_matches(candidate_matches)

    unmatched_images = [
        image_file.name
        for image_file, mask_matches in matched_pairs.items()
        if not mask_matches
    ]

    ambiguous_images = [
        image_file.name
        for image_file, mask_matches in matched_pairs.items()
        if len(mask_matches) > 1
    ]

    mask_to_images = {}
    for image_file, mask_matches in matched_pairs.items():
        for mask_file in mask_matches:
            mask_to_images.setdefault(mask_file, []).append(image_file)

    shared_masks = [
        mask_file.name
        for mask_file, linked_images in mask_to_images.items()
        if len(linked_images) > 1
    ]

    unmatched_masks = [
        mask_file.name
        for mask_file in mask_files
        if mask_file not in mask_to_images
    ]

    if unmatched_images or ambiguous_images or shared_masks or unmatched_masks:
        error_lines = ["Validation failed."]
        if unmatched_images:
            error_lines.append(
                "Images without a matching mask: " + ", ".join(unmatched_images[:10])
            )
        if ambiguous_images:
            error_lines.append(
                "Images with multiple possible masks after filtering: " + ", ".join(ambiguous_images[:10])
            )
        if shared_masks:
            error_lines.append(
                "Masks linked to multiple images after filtering: " + ", ".join(shared_masks[:10])
            )
        if unmatched_masks:
            error_lines.append(
                "Masks without a matching image: " + ", ".join(unmatched_masks[:10])
            )
        raise ValueError("\n".join(error_lines))

    return {
        image_file: mask_matches
        for image_file, mask_matches in matched_pairs.items()
        if mask_matches
    }


def _read_array(file_path):
    # Prefer tifffile for microscopy data; use imageio as a fallback.
    if _HAS_TIFFFILE:
        try:
            return tifffile.imread(file_path)
        except Exception:
            pass

    if _HAS_IMAGEIO:
        try:
            return iio.imread(file_path)
        except Exception:
            pass

    raise RuntimeError(
        "Could not read image data. Install tifffile (recommended) or imageio."
    )


def _write_array(file_path, array):
    if _HAS_TIFFFILE:
        try:
            tifffile.imwrite(str(file_path), array)
            return
        except Exception:
            pass

    if _HAS_IMAGEIO:
        iio.imwrite(str(file_path), array)
        return

    raise RuntimeError(
        "Could not write image data. Install tifffile (recommended) or imageio."
    )


def _read_array_shape(file_path):
    try:
        return _read_array(file_path).shape
    except Exception as exc:
        raise RuntimeError(
            "Could not read image data shape. Install tifffile (recommended) or imageio."
        ) from exc


def _default_axis_labels(ndim):
    # Default order follows common microscopy conventions.
    defaults_map = {
        2: ["Y", "X"],
        3: ["C", "Y", "X"],
        4: ["T", "C", "Y", "X"],
        5: ["T", "C", "Z", "Y", "X"],
    }
    if ndim in defaults_map:
        return defaults_map[ndim]
    return ["Other"] * ndim


def _build_mapping_ui(section_title, input_image_name, shape):
    axis_options = ["X", "Y", "Z", "T", "C", "Other"]
    defaults = _default_axis_labels(len(shape))
    dropdowns = []

    for axis_index, axis_size in enumerate(shape):
        default_label = defaults[axis_index] if axis_index < len(defaults) else "Other"
        dropdown = widgets.Dropdown(
            options=axis_options,
            value=default_label if default_label in axis_options else "Other",
            description=f"Dimension {axis_index} ({axis_size}):",
            layout=widgets.Layout(width="360px"),
            style={"description_width": "initial"},
        )
        dropdowns.append(dropdown)

    section = widgets.VBox([
        widgets.HTML(f"<h3><b>{section_title}</b></h3>"),
        widgets.HTML(f"Filename: {input_image_name}"),
        widgets.HTML(f"Shape: {tuple(shape)}"),
        widgets.VBox(dropdowns),
    ])
    return section, dropdowns


def _extract_mapping(dropdowns):
    return {dim_idx: dropdown.value for dim_idx, dropdown in enumerate(dropdowns)}


def _validate_mapping(mapping, label):
    assigned_labels = [axis_name for axis_name in mapping.values() if axis_name != "Other"]
    repeated = sorted({axis_name for axis_name in assigned_labels if assigned_labels.count(axis_name) > 1})
    if repeated:
        raise ValueError(
            f"{label} has duplicate axis assignments: {', '.join(repeated)}. "
            "Use each axis label at most once (except Other)."
        )


def _mapping_name_to_axis(mapping):
    return {axis_name: axis_index for axis_index, axis_name in mapping.items()}


def _reorder_with_spatial_first(array, mapping):
    name_to_axis = _mapping_name_to_axis(mapping)
    if "Y" not in name_to_axis or "X" not in name_to_axis:
        raise ValueError("Axis mapping must include Y and X.")

    y_axis = name_to_axis["Y"]
    x_axis = name_to_axis["X"]
    ndim = array.ndim

    ordered_axes = [y_axis, x_axis] + [
        axis_idx for axis_idx in range(ndim) if axis_idx not in (y_axis, x_axis)
    ]
    reordered = np.transpose(array, axes=ordered_axes)
    return reordered, ordered_axes


def _to_2d_from_mapping(array, mapping, channel_index=None, reduce_mode="first"):
    reordered, ordered_axes = _reorder_with_spatial_first(array, mapping)

    # Reduce non-spatial axes to one YX plane.
    for offset, original_axis in enumerate(ordered_axes[2:]):
        axis_pos = offset + 2
        axis_name = mapping.get(original_axis, "Other")

        if axis_name == "C" and channel_index is not None:
            use_idx = min(int(channel_index), reordered.shape[axis_pos] - 1)
            reordered = np.take(reordered, use_idx, axis=axis_pos)
        elif reduce_mode == "max":
            reordered = np.max(reordered, axis=axis_pos)
        else:
            reordered = np.take(reordered, 0, axis=axis_pos)

    return np.asarray(reordered)


def _to_2d_mask_from_mapping(array, mapping, channel_index=None):
    return _to_2d_from_mapping(array, mapping, channel_index=channel_index, reduce_mode="first")


def _label_binary_components(binary_mask):
    # Basic 4-connectivity labeling to split binary masks into individual objects.
    labels = np.zeros(binary_mask.shape, dtype=np.int32)
    label_id = 0
    h, w = binary_mask.shape

    for y in range(h):
        for x in range(w):
            if not binary_mask[y, x] or labels[y, x] != 0:
                continue
            label_id += 1
            stack = [(y, x)]
            labels[y, x] = label_id
            while stack:
                cy, cx = stack.pop()
                for ny, nx in ((cy - 1, cx), (cy + 1, cx), (cy, cx - 1), (cy, cx + 1)):
                    if 0 <= ny < h and 0 <= nx < w and binary_mask[ny, nx] and labels[ny, nx] == 0:
                        labels[ny, nx] = label_id
                        stack.append((ny, nx))

    return labels


def _touches_border_from_indices(ys, xs, h, w):
    return (
        (ys.min() == 0)
        or (xs.min() == 0)
        or (ys.max() == h - 1)
        or (xs.max() == w - 1)
    )


def _extract_square_patch(image_2d, obj_mask, square_size, apply_mask, center_cell):
    ys, xs = np.where(obj_mask)
    y0, y1 = int(ys.min()), int(ys.max()) + 1
    x0, x1 = int(xs.min()), int(xs.max()) + 1

    if center_cell:
        center_y = int(round((y0 + y1 - 1) / 2))
        center_x = int(round((x0 + x1 - 1) / 2))
        half_size = square_size // 2
        patch_y0 = center_y - half_size
        patch_x0 = center_x - half_size
        patch_y1 = patch_y0 + square_size
        patch_x1 = patch_x0 + square_size

        image_patch = np.zeros((square_size, square_size), dtype=image_2d.dtype)
        mask_patch = np.zeros((square_size, square_size), dtype=bool)

        src_y0 = max(0, patch_y0)
        src_x0 = max(0, patch_x0)
        src_y1 = min(image_2d.shape[0], patch_y1)
        src_x1 = min(image_2d.shape[1], patch_x1)

        dst_y0 = src_y0 - patch_y0
        dst_x0 = src_x0 - patch_x0
        dst_y1 = dst_y0 + (src_y1 - src_y0)
        dst_x1 = dst_x0 + (src_x1 - src_x0)

        image_patch[dst_y0:dst_y1, dst_x0:dst_x1] = image_2d[src_y0:src_y1, src_x0:src_x1]
        mask_patch[dst_y0:dst_y1, dst_x0:dst_x1] = obj_mask[src_y0:src_y1, src_x0:src_x1]
    else:
        image_patch = image_2d[y0:y1, x0:x1]
        mask_patch = obj_mask[y0:y1, x0:x1]

    if apply_mask:
        return np.where(mask_patch, image_patch, 0), mask_patch
    return image_patch, mask_patch


def _extract_square_patch_nd(image_spatial_first, obj_mask, square_size, apply_mask, center_cell):
    ys, xs = np.where(obj_mask)
    y0, y1 = int(ys.min()), int(ys.max()) + 1
    x0, x1 = int(xs.min()), int(xs.max()) + 1

    extra_shape = image_spatial_first.shape[2:]

    if center_cell:
        center_y = int(round((y0 + y1 - 1) / 2))
        center_x = int(round((x0 + x1 - 1) / 2))
        half_size = square_size // 2
        patch_y0 = center_y - half_size
        patch_x0 = center_x - half_size
        patch_y1 = patch_y0 + square_size
        patch_x1 = patch_x0 + square_size

        image_patch = np.zeros((square_size, square_size) + extra_shape, dtype=image_spatial_first.dtype)
        mask_patch = np.zeros((square_size, square_size), dtype=bool)

        src_y0 = max(0, patch_y0)
        src_x0 = max(0, patch_x0)
        src_y1 = min(image_spatial_first.shape[0], patch_y1)
        src_x1 = min(image_spatial_first.shape[1], patch_x1)

        dst_y0 = src_y0 - patch_y0
        dst_x0 = src_x0 - patch_x0
        dst_y1 = dst_y0 + (src_y1 - src_y0)
        dst_x1 = dst_x0 + (src_x1 - src_x0)

        image_patch[dst_y0:dst_y1, dst_x0:dst_x1, ...] = image_spatial_first[src_y0:src_y1, src_x0:src_x1, ...]
        mask_patch[dst_y0:dst_y1, dst_x0:dst_x1] = obj_mask[src_y0:src_y1, src_x0:src_x1]
    else:
        image_patch = image_spatial_first[y0:y1, x0:x1, ...]
        mask_patch = obj_mask[y0:y1, x0:x1]

    if apply_mask:
        mask_expanded = mask_patch[(...,) + (None,) * (image_patch.ndim - 2)]
        return np.where(mask_expanded, image_patch, 0), mask_patch
    return image_patch, mask_patch


def _resize_patch(patch, output_shape, method):
    # Preview helper for 2D arrays.
    target_height, target_width = output_shape
    if not _HAS_PIL:
        raise RuntimeError(
            "Pillow is required for resizing. Install it with: pip install pillow"
        )

    resample_map = {
        "nearest": Image.Resampling.NEAREST,
        "bilinear": Image.Resampling.BILINEAR,
        "bicubic": Image.Resampling.BICUBIC,
    }

    patch_array = np.asarray(patch)
    patch_min = float(np.min(patch_array))
    patch_max = float(np.max(patch_array))

    if patch_max > patch_min:
        scaled = ((patch_array - patch_min) / (patch_max - patch_min) * 255).astype(np.uint8)
        resized = Image.fromarray(scaled).resize(
            (target_width, target_height),
            resample=resample_map[method],
        )
        resized_array = np.asarray(resized).astype(np.float32) / 255.0
        return resized_array * (patch_max - patch_min) + patch_min

    scaled = np.zeros_like(patch_array, dtype=np.uint8)
    resized = Image.fromarray(scaled).resize(
        (target_width, target_height),
        resample=resample_map[method],
    )
    return np.asarray(resized, dtype=np.float32) + patch_min


def _resize_patch_keep_range(patch, output_shape, method):
    # Export helper: keep all extra dimensions and preserve original dtype/range semantics.
    target_height, target_width = output_shape
    patch = np.asarray(patch)

    if patch.shape[0] == target_height and patch.shape[1] == target_width:
        return patch

    order_map = {
        "nearest": 0,
        "bilinear": 1,
        "bicubic": 3,
    }
    interp_order = order_map[method]

    if _HAS_SCIPY:
        zoom_factors = [
            target_height / patch.shape[0],
            target_width / patch.shape[1],
        ] + [1.0] * (patch.ndim - 2)

        resized = scipy_zoom(
            patch,
            zoom=zoom_factors,
            order=interp_order,
            mode="nearest",
            prefilter=(interp_order > 1),
        )
    else:
        if method != "nearest":
            raise RuntimeError(
                "Bilinear/Bicubic resizing for multi-dimensional patches requires scipy. "
                "Install it with: pip install scipy"
            )

        # Nearest-neighbor fallback without extra dependencies.
        y_idx = np.floor(np.linspace(0, patch.shape[0] - 1, target_height)).astype(int)
        x_idx = np.floor(np.linspace(0, patch.shape[1] - 1, target_width)).astype(int)
        resized = patch[y_idx][:, x_idx, ...]

    if np.issubdtype(patch.dtype, np.integer):
        info = np.iinfo(patch.dtype)
        resized = np.clip(np.rint(resized), info.min, info.max).astype(patch.dtype)
    else:
        resized = resized.astype(patch.dtype, copy=False)

    return resized

widget_layout = widgets.Layout(width="1000px")
widget_style = {"description_width": "initial"}
correct_html = ("<span style='color:#0a7f2e;'>✅ ","</span>")
error_html = ("<span style='color:#b00020;'>❌ ","</span>")
warning_html = ("<span style='color:#b26a00;'>⚠️ ","</span>")

display(widgets.HTML(f"{correct_html[0]}<b>Import and utility functions correctly loaded.</b>{correct_html[1]}"))

## 1. Give you data paths

Please provide the paths toyour image and mask folders, and click "Validate paths". The tool will check that the paths exist and contain files, and that the files are in a supported format. If the validation is successful, you will be able to proceed to the next step.

In [ ]:
# @markdown Run this cell to show the folder selection widgets.

# Build and display selectors for the image and mask folders.
image_path_widget_ui, image_path_widget = get_path_widget(
    description="Images folder:",
    placeholder="e.g. C:/data/images",
    select="directory",
    widgets_layout=widget_layout,
    widgets_style=widget_style,
)

mask_path_widget_ui, mask_path_widget = get_path_widget(
    description="Masks folder:",
    placeholder="e.g. C:/data/masks",
    select="directory",
    widgets_layout=widget_layout,
    widgets_style=widget_style,
)

validate_button = widgets.Button(
    description="Validate folders",
    button_style="primary",
    icon="check",
    tooltip="Check that both folders are filled and filenames match",
    layout=widget_layout,
    widgets_style=widget_style,
)
validation_message = widgets.HTML()
validated_image_mask_pairs = None

def on_validate_button_clicked(_):
    global validated_image_mask_pairs

    # Reset/refresh validation state every time the user clicks validate.
    try:
        validated_image_mask_pairs = validate_image_and_mask_folders(
            image_path_widget.value,
            mask_path_widget.value,
        )
    except ValueError as exc:
        validated_image_mask_pairs = None
        validation_message.value = (
            f'{error_html[0]}{exc}{error_html[1]}'
        )
        return

    # Show summary and a few sample matches to make quick sanity checks easier.
    validation_message.value = (
        f'{correct_html[0]}<b>Validation successful:</b><br> '
        f'{len(validated_image_mask_pairs)} image files were matched to mask files.{correct_html[1]}'
    )

    validation_message.value += "<br><h3>Example matches:</h3>"
    validation_message.value += "<ul>"
    for image_file, mask_matches in list(validated_image_mask_pairs.items())[:5]:
        validation_message.value += f"<li>{image_file.name} -> {[mask_match.name for mask_match in mask_matches]}</li>"
    validation_message.value += "</ul>"

validate_button.on_click(on_validate_button_clicked)

title_widget = widgets.HTML("<h2>Please provide the folder paths</h2>")

display(widgets.VBox([title_widget, image_path_widget_ui, mask_path_widget_ui,
                      validate_button, validation_message]))

## 2. Configurate image processing

On this seconde step you will need to assign the semantic meaning of each dimension of your image and mask arrays. 

Then, based on the assigned axis labels, you will need to choose (in case you have multiple masks/channels) which one you want to use for the patching process.

In [ ]:
# @markdown Run this cell to show the axis mapping interface.

# Build an axis-configuration interface from one sample image and one sample mask.
if "validated_image_mask_pairs" not in globals() or not validated_image_mask_pairs:
    display(widgets.HTML(
        f"{error_html[0]}Please run the validation cell first and ensure it succeeds.{error_html[1]}"
    ))
else:
    # Use one representative pair to configure dimensional semantics for the whole dataset.
    sample_image_file = next(iter(validated_image_mask_pairs.keys()))
    sample_mask_file = validated_image_mask_pairs[sample_image_file][0]

    try:
        image_shape = _read_array_shape(sample_image_file)
        mask_shape = _read_array_shape(sample_mask_file)
    except RuntimeError as exc:
        display(widgets.HTML(f"{error_html[0]}{exc}{error_html[1]}"))
    else:
        # Build one dropdown per dimension so users can label each axis explicitly.
        image_section, image_dropdowns = _build_mapping_ui("Image axis mapping", sample_image_file.name, image_shape)
        mask_section, mask_dropdowns = _build_mapping_ui("Mask axis mapping", sample_mask_file.name, mask_shape)

        # Stretch each panel to fill its half of the available horizontal space.
        image_section.layout = widgets.Layout(flex="1 1 0", min_width="0")
        mask_section.layout = widgets.Layout(flex="1 1 0", min_width="0")

        save_axis_button = widgets.Button(
            description="Save axis mapping",
            button_style="primary",
            icon="save",
            tooltip="Validate and store the axis configuration",
            layout=widgets.Layout(width="99%")
        )
        axis_message = widgets.HTML()

        # This row holds the pretty mapping summary cards at full width.
        axis_mapping_message = widgets.HBox(
            [],
            layout=widgets.Layout(width="99%", align_items="stretch", justify_content="space-between"),
        )

        image_axis_mapping = None
        mask_axis_mapping = None

        def on_save_axis_mapping_clicked(_):
            global image_axis_mapping, mask_axis_mapping

            try:
                image_axis_mapping = _extract_mapping(image_dropdowns)
                mask_axis_mapping = _extract_mapping(mask_dropdowns)
                # Ensure axis labels are not duplicated (except "Other").
                _validate_mapping(image_axis_mapping, "Image mapping")
                _validate_mapping(mask_axis_mapping, "Mask mapping")
            except ValueError as exc:
                axis_message.value = f"{error_html[0]}{exc}{error_html[1]}"
                return

            axis_message.value = (
                f"{correct_html[0]}<b>Axis mapping saved successfully.</b>{correct_html[1]}"
            )

            image_items = "".join(
                f"<li><span style='color:#666;'>Dimension {dim_idx}</span>: <b>{axis_name}</b></li>"
                for dim_idx, axis_name in image_axis_mapping.items()
            )
            mask_items = "".join(
                f"<li><span style='color:#666;'>Dimension {dim_idx}</span>: <b>{axis_name}</b></li>"
                for dim_idx, axis_name in mask_axis_mapping.items()
            )

            image_card = widgets.HTML(
                value=(
                    "<div style='width:100%; box-sizing:border-box; padding:12px 14px; "
                    "border:1px solid #d9e2ec; border-radius:8px; background:#f8fbff;'>"
                    "<h4 style='margin:0 0 8px 0;'>Image mapping</h4>"
                    f"<ul style='margin:0; padding-left:18px;'>{image_items}</ul>"
                    "</div>"
                ),
                layout=widgets.Layout(flex="1 1 0", width="auto"),
            )

            mask_card = widgets.HTML(
                value=(
                    "<div style='width:100%; box-sizing:border-box; padding:12px 14px; "
                    "border:1px solid #d9e2ec; border-radius:8px; background:#f8fbff;'>"
                    "<h4 style='margin:0 0 8px 0;'>Mask mapping</h4>"
                    f"<ul style='margin:0; padding-left:18px;'>{mask_items}</ul>"
                    "</div>"
                ),
                layout=widgets.Layout(flex="1 1 0", width="auto"),
            )

            axis_mapping_message.children = [image_card, mask_card]

        save_axis_button.on_click(on_save_axis_mapping_clicked)

        # Section title and guidance shown above the side-by-side panels.
        title = widgets.HTML(
            "<h2 style='margin-bottom:4px;'>Configure axis mapping</h2>"
            "<p style='color:#555; margin-top:0;'>"
            "Assign a semantic label to each dimension of your image and mask arrays. Here it is a quick preview to help you understand the current order and size of dimensions."
            "<ul>"
            "<li><b>X</b> and <b>Y</b> should be assigned to the spatial dimensions (width and height).</li>"
            "<li><b>C</b> is typically used for channels in multi-channel images.</li>"
            "<li><b>Z</b> is typically used for depth in 3D stacks.</li>"
            "<li><b>T</b> is typically used for time in time-lapse data.</li>"
            "<li><b>Other</b> can be used for any dimension that does not fit the common categories, but try to label important ones correctly for best results.</li>"
            "</ul>"
            "</p>"
        )

        # Place image and mask panels side by side, each filling half the display.
        divider = widgets.HTML("<div style='border-left:1px solid #ddd; margin:0 16px;'></div>")
        panels = widgets.HBox(
            [image_section, divider, mask_section],
            layout=widgets.Layout(width="99%", align_items="flex-start"),
        )

        display(widgets.VBox([
            title,
            panels,
            save_axis_button,
            axis_message,
            axis_mapping_message,
        ], layout=widgets.Layout(width="99%")))

In [ ]:
# @markdown Run this cell to show the cell mask channel selection interface (if applicable).

# Choose which mask channel corresponds to the cell mask after axis mapping is saved.
if "mask_axis_mapping" not in globals() or not mask_axis_mapping:
    display(widgets.HTML(
        f"{error_html[0]}Please save the axis mapping first.{error_html[1]}"
    ))
elif "sample_mask_file" not in globals():
    display(widgets.HTML(
        f"{error_html[0]}Sample mask file is missing. Re-run the previous section.{error_html[1]}"
    ))
else:
    channel_axes = [
        axis_index
        for axis_index, axis_name in mask_axis_mapping.items()
        if axis_name == "C"
    ]

    if not channel_axes:
        selected_cell_mask_channel = None
        display(widgets.HTML(
            f"{correct_html[0]}No channel axis detected in masks. "
            f"Channel selection is not required.{correct_html[1]}"
        ))
    elif len(channel_axes) > 1:
        display(widgets.HTML(
            f"{error_html[0]}More than one channel axis is marked for masks. "
            f"Please fix mask axis mapping first.{error_html[1]}"
        ))
    else:
        channel_axis = channel_axes[0]

        try:
            mask_array = _read_array(sample_mask_file)
        except RuntimeError as exc:
            display(widgets.HTML(f"{error_html[0]}{exc}{error_html[1]}"))
        else:
            if channel_axis >= mask_array.ndim:
                display(widgets.HTML(
                    f"{error_html[0]}Channel axis {channel_axis} is out of bounds "
                    f"for mask shape {mask_array.shape}. Re-check axis mapping.{error_html[1]}"
                ))
            else:
                channel_first = np.moveaxis(mask_array, channel_axis, 0)
                n_channels = int(channel_first.shape[0])

                def _to_2d_preview(channel_data):
                    preview = np.asarray(channel_data)
                    if preview.ndim == 2:
                        return preview
                    if preview.ndim < 2:
                        return preview.reshape(1, -1)
                    # Collapse all non-spatial leading dims to one representative slice.
                    leading = int(np.prod(preview.shape[:-2]))
                    return preview.reshape((leading,) + preview.shape[-2:])[0]

                chooser_message = widgets.HTML()
                preview_message = widgets.HTML()
                preview_image = widgets.Image(format="png", layout=widgets.Layout(width="99%"))

                checkbox_rows = []
                channel_checkboxes = []

                for channel_index in range(n_channels):
                    checkbox = widgets.Checkbox(
                        value=(channel_index == 0),
                        description=f"Use channel {channel_index} as cell mask",
                        indent=False,
                        layout=widgets.Layout(width="99%"),
                    )
                    channel_checkboxes.append(checkbox)
                    checkbox_rows.append(widgets.HBox([checkbox], layout=widgets.Layout(width="99%")))

                # Keep checkbox behavior single-choice while preserving the checkbox UI.
                def _on_channel_toggle(change):
                    if change.get("name") != "value" or not change.get("new"):
                        return
                    source = change["owner"]
                    for checkbox in channel_checkboxes:
                        if checkbox is not source:
                            checkbox.value = False

                for checkbox in channel_checkboxes:
                    checkbox.observe(_on_channel_toggle, names="value")

                def _selected_channels():
                    return [
                        index
                        for index, checkbox in enumerate(channel_checkboxes)
                        if checkbox.value
                    ]

                save_channel_button = widgets.Button(
                    description="Save cell-mask channel",
                    button_style="primary",
                    icon="save",
                    layout=widgets.Layout(width="99%"),
                )

                selected_cell_mask_channel = None

                def _render_channel_previews():
                    if not _HAS_MPL:
                        preview_message.value = (
                            f"{error_html[0]}matplotlib is required for channel previews. "
                            f"Install it with: pip install matplotlib{error_html[1]}"
                        )
                        preview_image.value = b""
                        return

                    cols = min(3, n_channels)
                    rows = (n_channels + cols - 1) // cols
                    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
                    axes = np.array(axes).reshape(-1)

                    for channel_index in range(n_channels):
                        channel_image = _to_2d_preview(channel_first[channel_index])
                        axes[channel_index].imshow(channel_image, cmap="gray")
                        axes[channel_index].set_title(f"Channel {channel_index}")
                        axes[channel_index].axis("off")

                    for axis_index in range(n_channels, len(axes)):
                        axes[axis_index].axis("off")

                    fig.tight_layout()

                    buffer = io.BytesIO()
                    fig.savefig(buffer, format="png", bbox_inches="tight")
                    plt.close(fig)
                    preview_image.value = buffer.getvalue()
                    preview_message.value = ""

                def on_save_channel_clicked(_):
                    global selected_cell_mask_channel

                    chosen = _selected_channels()
                    if len(chosen) != 1:
                        chooser_message.value = (
                            f"{error_html[0]}Please select exactly one channel.{error_html[1]}"
                        )
                        return

                    selected_cell_mask_channel = chosen[0]
                    chooser_message.value = (
                        f"{correct_html[0]}Cell-mask channel saved: "
                        f"channel {selected_cell_mask_channel}{correct_html[1]}"
                    )

                save_channel_button.on_click(on_save_channel_clicked)
                _render_channel_previews()

                # Header and documentation for this step.
                title = widgets.HTML(
                    "<h2 style='margin-bottom:4px;'>Select mask channel</h2>"
                    "<p style='color:#555; margin-top:0;'>"
                    "Choose the channel that represents your cell mask labels. "
                    "Only one channel can be selected and saved."
                    "</p>"
                )

                file_info = widgets.HTML(
                    f"<span style='color:#555;'>Sample mask: <b>{sample_mask_file.name}</b> &nbsp;|&nbsp; "
                    f"Shape: <b>{mask_array.shape}</b> &nbsp;|&nbsp; "
                    f"Detected channel axis: <b>{channel_axis}</b></span>"
                )

                # Left card: rendered preview mosaic.
                preview_card = widgets.VBox(
                    [
                        widgets.HTML("<h3 style='margin:0 0 8px 0;'>Channel preview</h3>"),
                        preview_message,
                        preview_image,
                    ],
                    layout=widgets.Layout(
                        flex="1 1 0",
                        min_width="0",
                        width="auto",
                        padding="12px",
                        border="1px solid #d9e2ec",
                        border_radius="8px",
                    ),
                )

                # Right card: channel selection controls.
                channel_list_box = widgets.VBox(
                    checkbox_rows,
                    layout=widgets.Layout(
                        width="99%",
                        max_height="260px",
                        overflow_y="auto",
                        padding="2px 0",
                    ),
                )

                controls_card = widgets.VBox(
                    [
                        widgets.HTML("<h3 style='margin:0 0 8px 0;'>Choose one channel</h3>"),
                        channel_list_box,
                        save_channel_button,
                        chooser_message,
                    ],
                    layout=widgets.Layout(
                        flex="1 1 0",
                        min_width="0",
                        width="auto",
                        padding="12px",
                        border="1px solid #d9e2ec",
                        border_radius="8px",
                    ),
                )

                content = widgets.HBox(
                    [preview_card, controls_card],
                    layout=widgets.Layout(width="99%", align_items="stretch", justify_content="space-between"),
                )

                display(widgets.VBox([
                    title,
                    file_info,
                    content,
                ], layout=widgets.Layout(width="99%")))

## 3. Configure the cell patching

In [ ]:
# @markdown Run this cell to show the per-cell patch preview based on the current configuration.

# Preview per-cell patches from one image and one selected cell-mask channel.
if "validated_image_mask_pairs" not in globals() or not validated_image_mask_pairs:
    display(widgets.HTML(
        f"{error_html[0]}Please run validation first.{error_html[1]}"
    ))
elif "image_axis_mapping" not in globals() or not image_axis_mapping or "mask_axis_mapping" not in globals() or not mask_axis_mapping:
    display(widgets.HTML(
        f"{error_html[0]}Please save image and mask axis mapping first.{error_html[1]}"
    ))
else:
    if not _HAS_MPL:
        display(widgets.HTML(
            f"{error_html[0]}matplotlib is required for patch previews. "
            f"Install it with: pip install matplotlib{error_html[1]}"
        ))
    else:
        sample_image_file = globals().get("sample_image_file")
        sample_mask_file = globals().get("sample_mask_file")

        if sample_image_file is None or sample_mask_file is None:
            sample_image_file = next(iter(validated_image_mask_pairs.keys()))
            sample_mask_file = validated_image_mask_pairs[sample_image_file][0]

        can_continue = True
        channel_index = None
        mask_channel_axis = [
            axis_index
            for axis_index, axis_name in mask_axis_mapping.items()
            if axis_name == "C"
        ]

        if mask_channel_axis:
            if "selected_cell_mask_channel" not in globals() or selected_cell_mask_channel is None:
                display(widgets.HTML(
                    f"{error_html[0]}Please select and save the mask channel in the previous step.{error_html[1]}"
                ))
                can_continue = False
            else:
                channel_index = int(selected_cell_mask_channel)

        if can_continue:
            try:
                image_array = _read_array(sample_image_file)
                mask_array = _read_array(sample_mask_file)
                # For images, project non-spatial dimensions by max to include all channels/slices in the preview.
                image_2d = _to_2d_from_mapping(
                    image_array,
                    image_axis_mapping,
                    channel_index=None,
                    reduce_mode="max",
                )
                # For masks, keep the selected channel and one representative slice for other non-spatial dims.
                mask_2d = _to_2d_from_mapping(
                    mask_array,
                    mask_axis_mapping,
                    channel_index=channel_index,
                    reduce_mode="first",
                )
            except Exception as exc:
                display(widgets.HTML(f"{error_html[0]}{exc}{error_html[1]}"))
            else:
                if image_2d.shape != mask_2d.shape:
                    display(widgets.HTML(
                        f"{error_html[0]}Image and mask planes have different shapes: "
                        f"{image_2d.shape} vs {mask_2d.shape}. Please verify axis mapping.{error_html[1]}"
                    ))
                else:
                    mask_2d = np.asarray(mask_2d)
                    if np.issubdtype(mask_2d.dtype, np.integer) and np.max(mask_2d) > 1:
                        labeled = mask_2d.astype(np.int32)
                    else:
                        labeled = _label_binary_components(mask_2d > 0)

                    object_ids = [obj_id for obj_id in np.unique(labeled) if obj_id != 0]

                    mask_h, mask_w = labeled.shape
                    filtered_object_ids = []
                    skipped_border_touching = 0
                    for obj_id in object_ids:
                        ys, xs = np.where(labeled == obj_id)
                        if _touches_border_from_indices(ys, xs, mask_h, mask_w):
                            skipped_border_touching += 1
                        else:
                            filtered_object_ids.append(obj_id)

                    if not filtered_object_ids:
                        display(widgets.HTML(
                            f"{error_html[0]}No segmented objects were found after excluding border-touching masks.{error_html[1]}"
                        ))
                    else:
                        max_show = min(12, len(filtered_object_ids))
                        cols = min(4, max_show)
                        rows = (max_show + cols - 1) // cols

                        channel_info = (
                            f"using mask channel <b>{channel_index}</b>"
                            if channel_index is not None
                            else "no dedicated mask channel selected"
                        )

                        fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
                        axes = np.array(axes).reshape(-1)

                        for idx, obj_id in enumerate(filtered_object_ids[:max_show]):
                            obj_mask = labeled == obj_id
                            ys, xs = np.where(obj_mask)
                            y0, y1 = ys.min(), ys.max() + 1
                            x0, x1 = xs.min(), xs.max() + 1

                            image_crop = image_2d[y0:y1, x0:x1]
                            mask_crop = obj_mask[y0:y1, x0:x1]
                            masked_patch = np.where(mask_crop, image_crop, 0)

                            axes[idx].imshow(masked_patch, cmap="gray")
                            axes[idx].set_title(f"Cell {obj_id} | {masked_patch.shape}")
                            axes[idx].axis("off")

                        for idx in range(max_show, len(axes)):
                            axes[idx].axis("off")

                        fig.tight_layout()

                        buffer = io.BytesIO()
                        fig.savefig(buffer, format="png", bbox_inches="tight")
                        plt.close(fig)

                        preview_title = widgets.HTML(
                            "<h2 style='margin-bottom:4px;'>Preview per-cell patches</h2>"
                            "<p style='color:#555; margin-top:0;'>"
                            "This preview shows example cropped cell patches extracted from the sample image. "
                            "Border-touching objects are excluded, and the image view uses a max projection across non-spatial dimensions for readability."
                            "</p>"
                        )

                        preview_info = widgets.HTML(
                            f"<span style='color:#555;'>"
                            f"Sample image: <b>{sample_image_file.name}</b> &nbsp;|&nbsp; "
                            f"Sample mask: <b>{sample_mask_file.name}</b> &nbsp;|&nbsp; "
                            f"{channel_info}"
                            f"</span>"
                        )

                        summary_card = widgets.HTML(
                            value=(
                                "<div style='width:99%; box-sizing:border-box; padding:12px 14px; "
                                "border:1px solid #d9e2ec; border-radius:8px; background:#f8fbff;'>"
                                "<h3 style='margin:0 0 8px 0;'>Preview summary</h3>"
                                f"<ul style='margin:0; padding-left:18px;'>"
                                f"<li>Preview image shape: <b>{image_2d.shape}</b></li>"
                                f"<li>Displayed patches: <b>{max_show}</b></li>"
                                f"<li>Total segmented objects: <b>{len(object_ids)}</b></li>"
                                f"<li>Detected non-border objects: <b>{len(filtered_object_ids)}</b></li>"
                                f"<li>Border-touching objects skipped: <b>{skipped_border_touching}</b></li>"
                                f"</ul>"
                                "</div>"
                            ),
                            layout=widgets.Layout(width="99%"),
                        )

                        preview_image_widget = widgets.Image(
                            value=buffer.getvalue(),
                            format="png",
                            layout=widgets.Layout(width="99%"),
                        )

                        preview_card = widgets.VBox(
                            [
                                widgets.HTML("<h3 style='margin:0 0 8px 0;'>Per-cell patch preview</h3>"),
                                preview_image_widget,
                            ],
                            layout=widgets.Layout(
                                width="99%",
                                padding="12px",
                                border="1px solid #d9e2ec",
                                border_radius="8px",
                            ),
                        )

                        display(widgets.VBox([
                            preview_title,
                            preview_info,
                            summary_card,
                            preview_card,
                        ], layout=widgets.Layout(width="99%")))

In [ ]:
# @markdown Run this cell to show the results of scanning all masks for maximum cell bounding-box size.

# Scan all masks and compute the maximum cell bounding-box size across the dataset.
if "validated_image_mask_pairs" not in globals() or not validated_image_mask_pairs:
    display(widgets.HTML(
        f"{error_html[0]}Please run validation first.{error_html[1]}"
    ))
elif "mask_axis_mapping" not in globals() or not mask_axis_mapping:
    display(widgets.HTML(
        f"{error_html[0]}Please save mask axis mapping first.{error_html[1]}"
    ))
else:
    can_continue = True
    mask_channel_axes = [
        axis_idx for axis_idx, axis_name in mask_axis_mapping.items() if axis_name == "C"
    ]
    selected_channel = None
    if mask_channel_axes:
        if "selected_cell_mask_channel" not in globals() or selected_cell_mask_channel is None:
            display(widgets.HTML(
                f"{error_html[0]}Please select and save the cell-mask channel first.{error_html[1]}"
            ))
            can_continue = False
        else:
            selected_channel = int(selected_cell_mask_channel)

    if can_continue:
        from contextlib import contextmanager

        total_masks_to_scan = sum(len(mask_file_list) for mask_file_list in validated_image_mask_pairs.values())
        detected_scan_workers = max(1, min(total_masks_to_scan, int(os.cpu_count() or 1)))

        mask_tasks = [
            (image_file.name, str(mask_file), dict(mask_axis_mapping), selected_channel)
            for image_file, mask_file_list in validated_image_mask_pairs.items()
            for mask_file in mask_file_list
        ]

        def _scan_single_mask_task(task):
            image_name, mask_file_str, local_mask_axis_mapping, local_selected_channel = task
            mask_file = Path(mask_file_str)

            try:
                mask_array = _read_array(mask_file)
                mask_2d = _to_2d_mask_from_mapping(
                    mask_array,
                    local_mask_axis_mapping,
                    channel_index=local_selected_channel,
                )
            except Exception as exc:
                return {
                    "error": f"{image_name} / {mask_file.name}: {exc}",
                    "cells_scanned": 0,
                    "border_skipped": 0,
                    "max_height": 0,
                    "max_width": 0,
                    "max_height_example": None,
                    "max_width_example": None,
                }

            if mask_2d.ndim != 2:
                return {
                    "error": (
                        f"{image_name} / {mask_file.name}: expected 2D mask after axis reduction, "
                        f"got shape {mask_2d.shape}"
                    ),
                    "cells_scanned": 0,
                    "border_skipped": 0,
                    "max_height": 0,
                    "max_width": 0,
                    "max_height_example": None,
                    "max_width_example": None,
                }

            mask_2d = np.asarray(mask_2d)
            if np.issubdtype(mask_2d.dtype, np.integer) and np.max(mask_2d) > 1:
                labeled = mask_2d.astype(np.int32)
            else:
                labeled = _label_binary_components(mask_2d > 0)

            object_ids = [obj_id for obj_id in np.unique(labeled) if obj_id != 0]

            local_cells_scanned = 0
            local_border_skipped = 0
            local_max_height = 0
            local_max_width = 0
            local_max_height_example = None
            local_max_width_example = None

            mask_h, mask_w = labeled.shape
            for obj_id in object_ids:
                obj_mask = labeled == obj_id
                ys, xs = np.where(obj_mask)

                if _touches_border_from_indices(ys, xs, mask_h, mask_w):
                    local_border_skipped += 1
                    continue

                bbox_height = int(ys.max() - ys.min() + 1)
                bbox_width = int(xs.max() - xs.min() + 1)

                local_cells_scanned += 1

                if bbox_height > local_max_height:
                    local_max_height = bbox_height
                    local_max_height_example = (mask_file.name, int(obj_id), bbox_height, bbox_width)

                if bbox_width > local_max_width:
                    local_max_width = bbox_width
                    local_max_width_example = (mask_file.name, int(obj_id), bbox_height, bbox_width)

            return {
                "error": None,
                "cells_scanned": local_cells_scanned,
                "border_skipped": local_border_skipped,
                "max_height": local_max_height,
                "max_width": local_max_width,
                "max_height_example": local_max_height_example,
                "max_width_example": local_max_width_example,
            }

        @contextmanager
        def _joblib_progress(progress_widget):
            if not _HAS_JOBLIB:
                yield
                return

            class _WidgetBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
                def __call__(self, *args, **kwargs):
                    completed = min(progress_widget.max, progress_widget.value + self.batch_size)
                    progress_widget.value = completed
                    progress_widget.description = f"{completed} masks"
                    return super().__call__(*args, **kwargs)

            original_callback = joblib.parallel.BatchCompletionCallBack
            joblib.parallel.BatchCompletionCallBack = _WidgetBatchCompletionCallback
            try:
                yield
            finally:
                joblib.parallel.BatchCompletionCallBack = original_callback

        title = widgets.HTML(
            "<h2 style='margin-bottom:4px;'>Scan cell bounding boxes</h2>"
            "<p style='color:#555; margin-top:0;'>"
            "This step scans all validated masks to estimate the largest cell bounding box. "
            "Choose how many workers to use, then start the scan when you are ready."
            "</p>"
        )

        channel_info = (
            f"using mask channel <b>{selected_channel}</b>"
            if selected_channel is not None
            else "with no dedicated mask channel selection"
        )

        scan_dataset_info = widgets.HTML(
            f"<span style='color:#555;'>"
            f"Masks available: <b>{total_masks_to_scan}</b> &nbsp;|&nbsp; "
            f"Auto-detected max workers: <b>{detected_scan_workers}</b> &nbsp;|&nbsp; "
            f"{channel_info}"
            f"</span>"
        )

        scan_workers_help = widgets.HTML(
            value=(
                "<span style='color:#555;'>"
                f"You can use between <b>1</b> and <b>{detected_scan_workers}</b> worker(s), based on the number of masks and CPU cores detected."
                "</span>"
            )
        )

        scan_workers_widget = widgets.BoundedIntText(
            value=detected_scan_workers,
            min=1,
            max=detected_scan_workers,
            description="Workers:",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="220px"),
        )

        scan_button = widgets.Button(
            description="Scan masks",
            button_style="primary",
            icon="search",
            tooltip="Scan masks and estimate the maximum cell bounding-box size",
            layout=widgets.Layout(width="200px"),
        )

        scan_joblib_message = widgets.HTML()
        if not _HAS_JOBLIB and len(mask_tasks) > 1:
            scan_joblib_message.value = (
                f"{warning_html[0]}joblib is not installed; parallel scanning is unavailable. "
                f"Install with: pip install joblib{warning_html[1]}"
            )

        scan_progress_bar = widgets.IntProgress(
            value=0,
            min=0,
            max=max(1, total_masks_to_scan),
            description="0 masks",
            bar_style="info",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="700px", visibility="hidden"),
        )
        scan_progress_label = widgets.HTML()
        scan_message = widgets.HTML()
        scan_results_box = widgets.VBox([], layout=widgets.Layout(width="99%"))

        def on_scan_masks_clicked(_):
            global bbox_size_summary

            scan_button.disabled = True
            scan_message.value = "<span style='color:#777;'>Scanning - please wait...</span>"
            scan_results_box.children = []
            scan_progress_bar.value = 0
            scan_progress_bar.max = max(1, total_masks_to_scan)
            scan_progress_bar.description = "0 masks"
            scan_progress_bar.bar_style = "info"
            scan_progress_bar.layout.visibility = "visible"
            scan_progress_label.value = ""
            bbox_size_summary = None

            requested_workers = min(int(scan_workers_widget.value), detected_scan_workers)
            use_parallel = requested_workers > 1 and len(mask_tasks) > 1

            scan_mode = "sequential"
            worker_count_used = 1
            results = []
            error_lines = []
            parallel_fallback_reason = None

            try:
                if use_parallel and not _HAS_JOBLIB:
                    scan_message.value = (
                        f"{warning_html[0]}joblib is not installed; running sequentially instead. "
                        f"Install with: pip install joblib{warning_html[1]}"
                    )
                    use_parallel = False

                if use_parallel:
                    try:
                        from joblib import Parallel, delayed

                        scan_mode = "parallel"
                        worker_count_used = requested_workers
                        scan_progress_label.value = (
                            f"<span style='color:#555;'>Running in parallel with {worker_count_used} process(es)...</span>"
                        )

                        with _joblib_progress(scan_progress_bar):
                            results = Parallel(n_jobs=worker_count_used, backend="loky", prefer="processes")(
                                delayed(_scan_single_mask_task)(task) for task in mask_tasks
                            )
                    except Exception as exc:
                        parallel_fallback_reason = str(exc)
                        use_parallel = False
                        scan_mode = "sequential"
                        worker_count_used = 1
                        results = []

                if not use_parallel:
                    for idx, task in enumerate(mask_tasks, start=1):
                        scan_progress_label.value = (
                            f"<span style='color:#555;'>Scanning mask {idx} / {len(mask_tasks)}...</span>"
                        )
                        results.append(_scan_single_mask_task(task))
                        scan_progress_bar.value = idx
                        scan_progress_bar.description = f"{idx} masks"

                max_bbox_height = 0
                max_bbox_width = 0
                total_cells_scanned = 0
                skipped_border_touching_cells = 0
                max_height_example = None
                max_width_example = None

                for result in results:
                    if result["error"]:
                        error_lines.append(result["error"])

                    total_cells_scanned += int(result["cells_scanned"])
                    skipped_border_touching_cells += int(result["border_skipped"])

                    if int(result["max_height"]) > max_bbox_height:
                        max_bbox_height = int(result["max_height"])
                        max_height_example = result["max_height_example"]

                    if int(result["max_width"]) > max_bbox_width:
                        max_bbox_width = int(result["max_width"])
                        max_width_example = result["max_width_example"]

                max_bbox_square_size = max(max_bbox_height, max_bbox_width)

                if total_cells_scanned == 0:
                    scan_progress_bar.bar_style = "danger"
                    scan_message.value = (
                        f"{error_html[0]}No segmented cells were found across the masks after excluding border-touching cells.{error_html[1]}"
                    )
                    return

                bbox_size_summary = {
                    "max_height": max_bbox_height,
                    "max_width": max_bbox_width,
                    "max_square_size": max_bbox_square_size,
                    "total_cells_scanned": total_cells_scanned,
                    "skipped_border_touching_cells": skipped_border_touching_cells,
                    "max_height_example": max_height_example,
                    "max_width_example": max_width_example,
                    "channel_used": selected_channel,
                }

                scan_info = widgets.HTML(
                    f"<span style='color:#555;'>"
                    f"Masks scanned: <b>{total_masks_to_scan}</b> &nbsp;|&nbsp; "
                    f"Cells analyzed: <b>{total_cells_scanned}</b> &nbsp;|&nbsp; "
                    f"Run mode: <b>{scan_mode}</b> &nbsp;|&nbsp; "
                    f"Workers used: <b>{worker_count_used}</b> &nbsp;|&nbsp; "
                    f"{channel_info}"
                    f"</span>"
                )

                summary_items = [
                    f"<li>Cells scanned (excluding border-touching): <b>{total_cells_scanned}</b></li>",
                    f"<li>Cells skipped for touching border: <b>{skipped_border_touching_cells}</b></li>",
                    f"<li>Maximum bounding-box height: <b>{max_bbox_height}</b></li>",
                    f"<li>Maximum bounding-box width: <b>{max_bbox_width}</b></li>",
                    f"<li>Suggested square patch size: <b>{max_bbox_square_size}</b></li>",
                ]

                if max_height_example is not None:
                    summary_items.append(
                        f"<li>Max height example: <b>{max_height_example[0]}</b>, label <b>{max_height_example[1]}</b>, "
                        f"h=<b>{max_height_example[2]}</b>, w=<b>{max_height_example[3]}</b></li>"
                    )

                if max_width_example is not None:
                    summary_items.append(
                        f"<li>Max width example: <b>{max_width_example[0]}</b>, label <b>{max_width_example[1]}</b>, "
                        f"h=<b>{max_width_example[2]}</b>, w=<b>{max_width_example[3]}</b></li>"
                    )

                summary_card = widgets.HTML(
                    value=(
                        "<div style='width:100%; box-sizing:border-box; padding:12px 14px; "
                        "border:1px solid #d9e2ec; border-radius:8px; background:#f8fbff;'>"
                        "<h3 style='margin:0 0 8px 0;'>Bounding-box scan summary</h3>"
                        f"<ul style='margin:0; padding-left:18px;'>{''.join(summary_items)}</ul>"
                        "</div>"
                    ),
                    layout=widgets.Layout(width="99%"),
                )

                error_card = None
                if parallel_fallback_reason is not None or error_lines:
                    error_messages = []
                    if parallel_fallback_reason is not None:
                        error_messages.append(
                            f"<li>Parallel scan fallback reason: <b>{parallel_fallback_reason}</b></li>"
                        )
                    for line in error_lines[:10]:
                        error_messages.append(f"<li>{line}</li>")
                    if len(error_lines) > 10:
                        error_messages.append(f"<li>... and {len(error_lines) - 10} more.</li>")

                    error_card = widgets.HTML(
                        value=(
                            "<div style='width:100%; box-sizing:border-box; padding:12px 14px; "
                            "border:1px solid #f0d7a1; border-radius:8px; background:#fffaf0;'>"
                            "<h3 style='margin:0 0 8px 0;'>Warnings and skipped masks</h3>"
                            f"<ul style='margin:0; padding-left:18px;'>{''.join(error_messages)}</ul>"
                            "</div>"
                        ),
                        layout=widgets.Layout(width="99%"),
                    )

                output_widgets = [scan_info, summary_card]
                if error_card is not None:
                    output_widgets.append(error_card)
                scan_results_box.children = output_widgets

                scan_progress_bar.value = scan_progress_bar.max
                scan_progress_bar.description = f"{scan_progress_bar.max} masks"
                scan_progress_bar.bar_style = "danger" if error_lines else "success"
                scan_message.value = (
                    f"{correct_html[0]}Bounding-box scan complete.{correct_html[1]}"
                )
            finally:
                scan_progress_label.value = ""
                scan_button.disabled = False

        scan_button.on_click(on_scan_masks_clicked)

        controls_card = widgets.VBox(
            [
                widgets.HTML("<h3 style='margin:0 0 8px 0;'>Scan settings</h3>"),
                scan_workers_help,
                scan_workers_widget,
                scan_button,
                scan_progress_bar,
                scan_progress_label,
                scan_message,
            ],
            layout=widgets.Layout(
                width="99%",
                padding="12px",
                border="1px solid #d9e2ec",
                border_radius="8px",
            ),
        )

        display(widgets.VBox([
            title,
            scan_dataset_info,
            scan_joblib_message,
            controls_card,
            scan_results_box,
        ], layout=widgets.Layout(width="99%")))

In [ ]:
# @markdown Run this cell to show a preview of how the patch extraction parameters will affect one sample cell.

# Configure patch extraction parameters and preview the selected behavior on one sample cell.
if "validated_image_mask_pairs" not in globals() or not validated_image_mask_pairs:
    display(widgets.HTML(
        "<span style='color:#b00020;'>Please run validation first.</span>"
    ))
elif "image_axis_mapping" not in globals() or not image_axis_mapping or "mask_axis_mapping" not in globals() or not mask_axis_mapping:
    display(widgets.HTML(
        "<span style='color:#b00020;'>Please save image and mask axis mapping first.</span>"
    ))
elif "bbox_size_summary" not in globals() or not bbox_size_summary:
    display(widgets.HTML(
        "<span style='color:#b00020;'>Please run the bounding-box size scan first.</span>"
    ))
else:
    if not _HAS_MPL:
        display(widgets.HTML(
            "<span style='color:#b00020;'>matplotlib is required for the preview. Install it with: pip install matplotlib</span>"
        ))
    else:
        title = widgets.HTML(
            "<h2 style='margin-bottom:4px;'>Configure patch extraction parameters</h2>"
            "<p style='color:#555; margin-top:0;'>"
            "Choose how cell patches should be cropped and resized, then validate the configuration on one sample cell before running the full export."
            "</p>"
        )

        sample_patch_size = int(bbox_size_summary["max_square_size"])
        total_cells_scanned = int(bbox_size_summary.get("total_cells_scanned", 0))
        skipped_border_touching_cells = int(bbox_size_summary.get("skipped_border_touching_cells", 0))

        patch_dataset_info = widgets.HTML(
            f"<span style='color:#555;'>"
            f"Suggested square patch size: <b>{sample_patch_size}</b> &nbsp;|&nbsp; "
            f"Cells measured during scan: <b>{total_cells_scanned}</b> &nbsp;|&nbsp; "
            f"Border-touching cells skipped during scan: <b>{skipped_border_touching_cells}</b>"
            f"</span>"
        )

        # User-facing controls for extraction behavior and final resize.
        apply_mask_widget = widgets.Checkbox(
            value=True,
            description="Apply mask to patch",
            indent=False,
            layout=widgets.Layout(width="99%"),
        )
        center_cell_widget = widgets.Checkbox(
            value=True,
            description="Center cell in square patch",
            indent=False,
            layout=widgets.Layout(width="99%"),
        )
        resize_width_widget = widgets.BoundedIntText(
            value=sample_patch_size,
            min=1,
            max=4096,
            description="Resize width (X):",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="260px"),
        )
        resize_height_widget = widgets.BoundedIntText(
            value=sample_patch_size,
            min=1,
            max=4096,
            description="Resize height (Y):",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="260px"),
        )
        resize_method_widget = widgets.Dropdown(
            options=[
                ("Nearest", "nearest"),
                ("Bilinear", "bilinear"),
                ("Bicubic", "bicubic"),
            ],
            value="nearest",
            description="Resize method:",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="260px"),
        )
        validate_patch_params_button = widgets.Button(
            description="Validate patch parameters",
            button_style="primary",
            icon="check",
            layout=widgets.Layout(width="220px"),
        )
        patch_params_message = widgets.HTML()
        patch_params_preview_message = widgets.HTML()
        patch_params_preview_image = widgets.Image(
            format="png",
            layout=widgets.Layout(width="99%"),
        )

        patch_parameters = None

        def _normalize_preview(preview_array):
            preview_array = np.asarray(preview_array, dtype=np.float32)
            preview_min = float(np.min(preview_array))
            preview_max = float(np.max(preview_array))
            if preview_max > preview_min:
                return (preview_array - preview_min) / (preview_max - preview_min)
            return np.zeros_like(preview_array, dtype=np.float32)

        def _render_spatial_first_preview(array_spatial_first):
            # Always max-project every non-spatial axis (including channel) to ensure a visible 2D preview.
            preview = np.asarray(array_spatial_first)
            while preview.ndim > 2:
                preview = np.max(preview, axis=-1)
            return _normalize_preview(preview), "gray"

        def on_validate_patch_params_clicked(_):
            global patch_parameters

            patch_params_preview_message.value = ""
            patch_params_preview_image.value = b""

            # Persist the chosen extraction settings for the final batch run.
            patch_parameters = {
                "apply_mask": bool(apply_mask_widget.value),
                "center_cell": bool(center_cell_widget.value),
                "square_patch_size": int(bbox_size_summary["max_square_size"]),
                "resize_shape": (int(resize_height_widget.value), int(resize_width_widget.value)),
                "resize_method": str(resize_method_widget.value),
                "skip_border_touching": True,
            }

            try:
                # Validate on one sample pair so failures are caught early.
                sample_image_file = next(iter(validated_image_mask_pairs.keys()))
                sample_mask_file = validated_image_mask_pairs[sample_image_file][0]
                image_array = _read_array(sample_image_file)
                mask_array = _read_array(sample_mask_file)

                selected_channel = None
                if any(axis_name == "C" for axis_name in mask_axis_mapping.values()):
                    if "selected_cell_mask_channel" not in globals() or selected_cell_mask_channel is None:
                        raise ValueError("Please select and save the mask channel before validating patch parameters.")
                    selected_channel = int(selected_cell_mask_channel)

                image_spatial_first, ordered_axes = _reorder_with_spatial_first(
                    image_array,
                    image_axis_mapping,
                )
                inverse_axes = np.argsort(ordered_axes)

                mask_2d = _to_2d_from_mapping(
                    mask_array,
                    mask_axis_mapping,
                    channel_index=selected_channel,
                    reduce_mode="first",
                )

                if image_spatial_first.shape[:2] != tuple(mask_2d.shape):
                    raise ValueError(
                        f"Image and mask preview shapes do not match: {image_spatial_first.shape[:2]} vs {mask_2d.shape}."
                    )

                # Keep existing labels if the mask is already labeled; otherwise split connected components.
                if np.issubdtype(mask_2d.dtype, np.integer) and np.max(mask_2d) > 1:
                    labeled = mask_2d.astype(np.int32)
                else:
                    labeled = _label_binary_components(mask_2d > 0)

                object_ids = [obj_id for obj_id in np.unique(labeled) if obj_id != 0]
                if not object_ids:
                    raise ValueError("No segmented cells were found in the sample mask.")

                mask_h, mask_w = labeled.shape
                filtered_object_ids = []
                skipped_border_touching = 0
                for obj_id in object_ids:
                    ys, xs = np.where(labeled == obj_id)
                    if _touches_border_from_indices(ys, xs, mask_h, mask_w):
                        skipped_border_touching += 1
                    else:
                        filtered_object_ids.append(obj_id)

                if not filtered_object_ids:
                    raise ValueError(
                        "No segmented cells remain after excluding border-touching masks in the sample preview."
                    )

                preview_object_id = int(filtered_object_ids[0])
                preview_object_mask = labeled == preview_object_id
                preview_patch_spatial_first, _ = _extract_square_patch_nd(
                    image_spatial_first,
                    preview_object_mask,
                    patch_parameters["square_patch_size"],
                    patch_parameters["apply_mask"],
                    patch_parameters["center_cell"],
                )
                resized_preview_patch_spatial_first = _resize_patch_keep_range(
                    preview_patch_spatial_first,
                    patch_parameters["resize_shape"],
                    patch_parameters["resize_method"],
                )

                full_image_preview, full_image_cmap = _render_spatial_first_preview(
                    image_spatial_first,
                )
                preview_patch_display, preview_patch_cmap = _render_spatial_first_preview(
                    preview_patch_spatial_first,
                )
                resized_patch_display, resized_patch_cmap = _render_spatial_first_preview(
                    resized_preview_patch_spatial_first,
                )

                preview_patch_original_order = np.transpose(
                    preview_patch_spatial_first,
                    axes=inverse_axes,
                )
                resized_preview_patch_original_order = np.transpose(
                    resized_preview_patch_spatial_first,
                    axes=inverse_axes,
                )
            except Exception as exc:
                patch_parameters = None
                patch_params_message.value = f"<pre style='color:#b00020; margin:0;'>{exc}</pre>"
                return

            patch_params_message.value = (
                f"{correct_html[0]}Patch parameters saved successfully.{correct_html[1]}"
            )

            patch_params_preview_message.value = (
                "<div style='width:100%; box-sizing:border-box; padding:12px 14px; "
                "border:1px solid #d9e2ec; border-radius:8px; background:#f8fbff;'>"
                "<h3 style='margin:0 0 8px 0;'>Validation summary</h3>"
                f"<ul style='margin:0; padding-left:18px;'>"
                f"<li>Apply mask: <b>{patch_parameters['apply_mask']}</b></li>"
                f"<li>Center cell: <b>{patch_parameters['center_cell']}</b></li>"
                f"<li>Skip border-touching cells: <b>{patch_parameters['skip_border_touching']}</b></li>"
                f"<li>Square patch size: <b>{patch_parameters['square_patch_size']}</b></li>"
                f"<li>Resize shape (Y, X): <b>{patch_parameters['resize_shape']}</b></li>"
                f"<li>Resize method: <b>{patch_parameters['resize_method']}</b></li>"
                f"<li>Sample border-touching cells skipped: <b>{skipped_border_touching}</b></li>"
                f"<li>Preview object id: <b>{preview_object_id}</b></li>"
                f"<li>Raw patch shape preserved for export: <b>{tuple(preview_patch_original_order.shape)}</b></li>"
                f"<li>Resized patch shape preserved for export: <b>{tuple(resized_preview_patch_original_order.shape)}</b></li>"
                f"</ul>"
                "</div>"
            )

            # Visual validation uses rendered max-projection views, while extraction/resizing preserves full multi-dimensional patches.
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            axes[0].imshow(full_image_preview, cmap=full_image_cmap)
            axes[0].set_title("Full image preview")
            axes[0].axis("off")

            axes[1].imshow(preview_object_mask, cmap="gray")
            axes[1].set_title(f"Sample cell mask (id {preview_object_id})")
            axes[1].axis("off")

            axes[2].imshow(preview_patch_display, cmap=preview_patch_cmap)
            axes[2].set_title("Patch before resize")
            axes[2].axis("off")

            axes[3].imshow(resized_patch_display, cmap=resized_patch_cmap)
            axes[3].set_title("Patch after resize")
            axes[3].axis("off")

            fig.tight_layout()
            buffer = io.BytesIO()
            fig.savefig(buffer, format="png", bbox_inches="tight")
            plt.close(fig)
            patch_params_preview_image.value = buffer.getvalue()

        validate_patch_params_button.on_click(on_validate_patch_params_clicked)

        extraction_options = widgets.VBox(
            [
                widgets.HTML("<h4 style='margin:0 0 8px 0;'>Extraction behavior</h4>"),
                apply_mask_widget,
                center_cell_widget,
            ],
            layout=widgets.Layout(
                width="99%",
                padding="0",
                margin="0 0 12px 0",
            ),
        )

        resize_controls = widgets.VBox(
            [
                widgets.HTML("<h4 style='margin:0 0 8px 0;'>Resize output</h4>"),
                widgets.HBox(
                    [resize_width_widget, resize_height_widget, resize_method_widget],
                    layout=widgets.Layout(width="99%", flex_flow="row wrap", gap="12px"),
                ),
            ],
            layout=widgets.Layout(width="99%"),
        )

        controls_card = widgets.VBox(
            [
                widgets.HTML("<h3 style='margin:0 0 8px 0;'>Patch settings</h3>"),
                widgets.HTML(
                    "<p style='color:#555; margin:0 0 12px 0;'>"
                    "Use the dataset scan recommendation as a starting point, then validate the exact crop and resize behavior on a sample cell."
                    "</p>"
                ),
                extraction_options,
                resize_controls,
                validate_patch_params_button,
                patch_params_message,
            ],
            layout=widgets.Layout(
                width="99%",
                padding="12px",
                border="1px solid #d9e2ec",
                border_radius="8px",
            ),
        )

        preview_card = widgets.VBox(
            [
                widgets.HTML("<h3 style='margin:0 0 8px 0;'>Preview and validation</h3>"),
                patch_params_preview_message,
                patch_params_preview_image,
            ],
            layout=widgets.Layout(
                width="99%",
                padding="12px",
                border="1px solid #d9e2ec",
                border_radius="8px",
            ),
        )

        display(widgets.VBox([
            title,
            patch_dataset_info,
            controls_card,
            preview_card,
        ], layout=widgets.Layout(width="99%")))

## 4. Run the patching

In [ ]:
# @markdown Run this cell to show the final patch extraction interface, which will run the extraction and save patches to disk for all validated image/mask pairs.

 # Run the patching on all validated image/mask pairs and save per-cell patches to disk.
if "validated_image_mask_pairs" not in globals() or not validated_image_mask_pairs:
    display(widgets.HTML(
        "<span style='color:#b00020;'>Please run validation first.</span>"
    ))
elif "image_axis_mapping" not in globals() or not image_axis_mapping \
        or "mask_axis_mapping" not in globals() or not mask_axis_mapping:
    display(widgets.HTML(
        "<span style='color:#b00020;'>Please save image and mask axis mapping first.</span>"
    ))
elif "patch_parameters" not in globals() or patch_parameters is None:
    display(widgets.HTML(
        "<span style='color:#b00020;'>Please configure and validate patch parameters first.</span>"
    ))
else:
    output_path_widget_ui, output_path_widget = get_path_widget(
        description="Output folder:",
        placeholder="e.g. C:/data/patches",
        select="directory",
        widgets_layout=widgets.Layout(width="1000px"),
    )

    # Auto-pick a safe worker ceiling based on available cores and image count.
    n_images = len(validated_image_mask_pairs)
    total_masks = sum(len(mask_file_list) for mask_file_list in validated_image_mask_pairs.values())
    detected_cpu_count = int(os.cpu_count() or 1)
    max_recommended_workers = max(1, min(n_images, detected_cpu_count))
    requires_mask_channel = any(axis_name == "C" for axis_name in mask_axis_mapping.values())

    title = widgets.HTML(
        "<h2 style='margin-bottom:4px;'>Run patch extraction</h2>"
        "<p style='color:#555; margin-top:0;'>"
        "Configure the output location and worker count, then run extraction over all validated image/mask pairs."
        "</p>"
    )

    channel_requirement_text = (
        "cell-mask channel selection required" if requires_mask_channel else "no dedicated mask channel required"
    )
    run_dataset_info = widgets.HTML(
        f"<span style='color:#555;'>"
        f"Images to process: <b>{n_images}</b> &nbsp;|&nbsp; "
        f"Masks linked: <b>{total_masks}</b> &nbsp;|&nbsp; "
        f"Auto-detected max workers: <b>{max_recommended_workers}</b> &nbsp;|&nbsp; "
        f"{channel_requirement_text}"
        f"</span>"
    )

    workers_widget = widgets.BoundedIntText(
        value=max_recommended_workers,
        min=1,
        max=max_recommended_workers,
        description="Workers:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="220px"),
    )

    workers_help = widgets.HTML(
        value=(
            "<span style='color:#555;'>"
            f"Recommended upper bound: <b>{max_recommended_workers}</b> worker(s) "
            f"(CPU cores detected: {detected_cpu_count}, image tasks: {n_images})."
            "</span>"
        )
    )

    run_button = widgets.Button(
        description="Run patching",
        button_style="success",
        icon="play",
        tooltip="Extract and save cell patches for all images",
        layout=widgets.Layout(width="200px"),
    )
    run_message = widgets.HTML()

    joblib_message = widgets.HTML()
    if not _HAS_JOBLIB:
        joblib_message.value = (
            "<span style='color:#b26a00;'>joblib is not installed; parallel processing is unavailable. "
            "Install with: pip install joblib</span>"
        )

    progress_bar = widgets.IntProgress(
        value=0,
        min=0,
        max=max(1, len(validated_image_mask_pairs)),
        description="0 images",
        bar_style="info",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="700px", visibility="hidden"),
    )
    progress_label = widgets.HTML()

    def _process_single_image_task(task):
        # One task = one image plus all matched masks. This is process-safe for joblib.
        image_file_str, mask_file_strs, output_folder_str, selected_channel_value, local_patch_parameters = task

        image_file = Path(image_file_str)
        output_folder = Path(output_folder_str)

        total_patches_saved = 0
        total_border_skipped = 0
        error_lines = []

        try:
            image_array = _read_array(image_file)
            image_spatial_first, ordered_axes = _reorder_with_spatial_first(
                image_array,
                image_axis_mapping,
            )
            inverse_axes = np.argsort(ordered_axes)

            for mask_file_str in mask_file_strs:
                mask_file = Path(mask_file_str)
                try:
                    mask_array = _read_array(mask_file)

                    # Use mask 2D only to define object positions.
                    mask_2d = _to_2d_from_mapping(
                        mask_array,
                        mask_axis_mapping,
                        channel_index=selected_channel_value,
                        reduce_mode="first",
                    )

                    if image_spatial_first.shape[:2] != tuple(mask_2d.shape):
                        raise ValueError(
                            f"Image/mask spatial mismatch: {image_spatial_first.shape[:2]} vs {mask_2d.shape}"
                        )

                    mask_2d = np.asarray(mask_2d)
                    if np.issubdtype(mask_2d.dtype, np.integer) and np.max(mask_2d) > 1:
                        labeled = mask_2d.astype(np.int32)
                    else:
                        labeled = _label_binary_components(mask_2d > 0)

                    object_ids = [obj_id for obj_id in np.unique(labeled) if obj_id != 0]

                    mask_h, mask_w = labeled.shape
                    for obj_id in object_ids:
                        obj_mask = labeled == obj_id
                        ys, xs = np.where(obj_mask)

                        # Exclude cells touching any border of the mask/image plane.
                        if _touches_border_from_indices(ys, xs, mask_h, mask_w):
                            total_border_skipped += 1
                            continue

                        # Extract patch in spatial-first order to preserve all non-spatial dimensions.
                        patch_spatial_first, _ = _extract_square_patch_nd(
                            image_spatial_first,
                            obj_mask,
                            local_patch_parameters["square_patch_size"],
                            local_patch_parameters["apply_mask"],
                            local_patch_parameters["center_cell"],
                        )

                        # Resize only Y/X; keep extra dimensions unchanged.
                        resized_spatial_first = _resize_patch_keep_range(
                            patch_spatial_first,
                            local_patch_parameters["resize_shape"],
                            local_patch_parameters["resize_method"],
                        )

                        # Restore original axis order before saving.
                        resized_original_order = np.transpose(resized_spatial_first, axes=inverse_axes)

                        output_filename = f"{image_file.stem}_cell{int(obj_id):04d}.tif"
                        output_file = output_folder / output_filename
                        _write_array(output_file, resized_original_order)

                        total_patches_saved += 1

                except Exception as exc:
                    error_lines.append(f"{image_file.name} / {mask_file.name}: {exc}")

        except Exception as exc:
            error_lines.append(f"{image_file.name}: {exc}")

        return {
            "saved": total_patches_saved,
            "border_skipped": total_border_skipped,
            "errors": error_lines,
            "image_name": image_file.name,
        }

    def on_run_patching_clicked(_):
        # Initialize run state and progress indicators.
        run_button.disabled = True
        run_message.value = "<span style='color:#777;'>Running - please wait...</span>"
        progress_bar.value = 0
        progress_bar.max = max(1, len(validated_image_mask_pairs))
        progress_bar.description = "0 images"
        progress_bar.bar_style = "info"
        progress_bar.layout.visibility = "visible"
        progress_label.value = ""

        output_path_str = output_path_widget.value.strip()
        if not output_path_str:
            run_message.value = (
                f"{warning_html[0]}Please provide an output folder path.{warning_html[1]}"
            )
            run_button.disabled = False
            progress_bar.layout.visibility = "hidden"
            return

        output_folder = Path(output_path_str).expanduser()
        try:
            output_folder.mkdir(parents=True, exist_ok=True)
        except Exception as exc:
            run_message.value = (
                f"{warning_html[0]}Could not create output folder: {exc}{warning_html[1]}"
            )
            run_button.disabled = False
            progress_bar.layout.visibility = "hidden"
            return

        selected_channel = None
        if any(axis_name == "C" for axis_name in mask_axis_mapping.values()):
            if "selected_cell_mask_channel" not in globals() or selected_cell_mask_channel is None:
                run_message.value = (
                    f"{warning_html[0]}Please select and save the mask channel first.{warning_html[1]}"
                )
                run_button.disabled = False
                progress_bar.layout.visibility = "hidden"
                return
            selected_channel = int(selected_cell_mask_channel)

        # Create serializable tasks to support process-based execution.
        tasks = [
            (
                str(image_file),
                [str(mask_file) for mask_file in mask_file_list],
                str(output_folder),
                selected_channel,
                dict(patch_parameters),
            )
            for image_file, mask_file_list in validated_image_mask_pairs.items()
        ]

        n_workers = min(int(workers_widget.value), max_recommended_workers)
        use_parallel = n_workers > 1 and len(tasks) > 1

        total_patches_saved = 0
        total_border_skipped = 0
        error_lines = []

        if use_parallel and not _HAS_JOBLIB:
            run_message.value = (
                f"{warning_html[0]}joblib is not installed; running sequentially. "
                f"Install with: pip install joblib{warning_html[1]}"
            )
            use_parallel = False

        if use_parallel:
            try:
                from joblib import Parallel, delayed
                progress_label.value = (
                    f"<span style='color:#555;'>Running in parallel with {n_workers} process(es) (one task per image)...</span>"
                )

                results = Parallel(n_jobs=n_workers, backend="loky", prefer="processes")(
                    delayed(_process_single_image_task)(task) for task in tasks
                )

                # Aggregate worker results and update visible progress by image.
                for idx, result in enumerate(results, start=1):
                    total_patches_saved += int(result["saved"] )
                    total_border_skipped += int(result["border_skipped"] )
                    error_lines.extend(result["errors"])

                    progress_bar.value = idx
                    progress_bar.description = f"{idx} images"
            except Exception as exc:
                # Fall back to sequential mode if process backend is unavailable.
                progress_label.value = (
                    f"{warning_html[0]}Parallel processing is unavailable in this environment. "
                    f"Falling back to sequential mode.{warning_html[1]}"
                )
                error_lines.append(f"Parallel fallback reason: {exc}")
                use_parallel = False

        if not use_parallel:
            for idx, task in enumerate(tasks, start=1):
                progress_label.value = (
                    f"<span style='color:#555;'>Processing image {idx} / {len(tasks)}...</span>"
                )
                result = _process_single_image_task(task)

                total_patches_saved += int(result["saved"] )
                total_border_skipped += int(result["border_skipped"] )
                error_lines.extend(result["errors"])

                progress_bar.value = idx
                progress_bar.description = f"{idx} images"

        progress_bar.value = progress_bar.max
        progress_bar.bar_style = "danger" if error_lines else "success"
        progress_label.value = ""

        # Final run summary includes successful outputs, skipped border-touching cells, and sampled errors.
        summary = (
            f"{correct_html[0]}"
            f"Patching complete. {total_patches_saved} patch(es) saved to "
            f"<b>{output_folder}</b>. "
            f"Skipped {total_border_skipped} border-touching object(s)."
            f"{correct_html[1]}"
        )
        if error_lines:
            summary += (
                f"{warning_html[0]}<br>{len(error_lines)} error(s) occurred:{warning_html[1]}"
            )
            for line in error_lines[:10]:
                summary += f"<br>{warning_html[0]}{line}{warning_html[1]}"
            if len(error_lines) > 10:
                summary += f"<br>{warning_html[0]}... and {len(error_lines) - 10} more.{warning_html[1]}"

        run_message.value = summary
        run_button.disabled = False

    run_button.on_click(on_run_patching_clicked)

    controls_card = widgets.VBox(
        [
            widgets.HTML("<h3 style='margin:0 0 8px 0;'>Run settings</h3>"),
            output_path_widget_ui,
            joblib_message,
            workers_help,
            workers_widget,
            run_button,
        ],
        layout=widgets.Layout(
            width="99%",
            padding="12px",
            border="1px solid #d9e2ec",
            border_radius="8px",
        ),
    )

    results_card = widgets.VBox(
        [
            widgets.HTML("<h3 style='margin:0 0 8px 0;'>Run progress and results</h3>"),
            progress_bar,
            progress_label,
            run_message,
        ],
        layout=widgets.Layout(
            width="99%",
            padding="12px",
            border="1px solid #d9e2ec",
            border_radius="8px",
        ),
    )

    display(widgets.VBox([
        title,
        run_dataset_info,
        controls_card,
        results_card,
    ], layout=widgets.Layout(width="99%")))
